In [2]:
!pip install pandas numpy scikit-learn streamlit requests

In [113]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import requests

In [114]:
import pandas as pd

movies = pd.read_csv('ml-latest-small/movies.csv')

ratings = pd.read_csv('ml-latest-small/ratings.csv')

tags = pd.read_csv('ml-latest-small/tags.csv')

links = pd.read_csv('ml-latest-small/links.csv')

In [115]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [116]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [117]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [118]:
print(movies.shape)
print(ratings.shape)
print(tags.shape)

(9742, 3)
(100836, 4)
(3683, 4)


In [119]:
movie_data = pd.merge(ratings, movies, on='movieId')
movie_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [120]:
movie_rating_count = movie_data.groupby('title')['rating'].count().reset_index()
movie_rating_count.rename(columns={'rating': 'num_ratings'}, inplace=True)

In [121]:
popular_movies = movie_rating_count[movie_rating_count['num_ratings'] >= 50]

In [122]:
movie_data = movie_data.merge(popular_movies, on='title')

In [123]:
user_movie_matrix = movie_data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

In [124]:
user_movie_matrix.fillna(0, inplace=True)

In [125]:
movie_similarity = cosine_similarity(user_movie_matrix.T)

In [126]:
similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_matrix.columns,
    columns=user_movie_matrix.columns
)

In [127]:
def recommend_movies(movie_name, top_n=10):
    
    similar_scores = similarity_df[movie_name].sort_values(ascending=False)
    
    recommendations = similar_scores.iloc[1:top_n+1]
    
    return recommendations

In [128]:
recommend_movies('Toy Story (1995)')

title
Toy Story 2 (1999)                                   0.572601
Jurassic Park (1993)                                 0.565637
Independence Day (a.k.a. ID4) (1996)                 0.564262
Star Wars: Episode IV - A New Hope (1977)            0.557388
Forrest Gump (1994)                                  0.547096
Lion King, The (1994)                                0.541145
Star Wars: Episode VI - Return of the Jedi (1983)    0.541089
Mission: Impossible (1996)                           0.538913
Groundhog Day (1993)                                 0.534169
Back to the Future (1985)                            0.530381
Name: Toy Story (1995), dtype: float64

In [129]:
movies['genres'] = movies['genres'].str.replace('|', ' ')

In [130]:
tfidf = TfidfVectorizer(stop_words='english')

In [131]:
tfidf_matrix = tfidf.fit_transform(movies['genres'])

In [132]:
content_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [133]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [134]:
def content_recommendations(title, top_n=10):
    
    idx = indices[title]
    
    sim_scores = list(enumerate(content_similarity[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies['title'].iloc[movie_indices]

In [135]:
content_recommendations('Toy Story (1995)')

1706                                          Antz (1998)
2355                                   Toy Story 2 (1999)
2809       Adventures of Rocky and Bullwinkle, The (2000)
3000                     Emperor's New Groove, The (2000)
3568                                Monsters, Inc. (2001)
6194                                     Wild, The (2006)
6486                               Shrek the Third (2007)
6948                       Tale of Despereaux, The (2008)
7760    Asterix and the Vikings (Astérix et les Viking...
8219                                         Turbo (2013)
Name: title, dtype: object

In [136]:
def hybrid_recommendation(movie_name):
    
    collaborative = recommend_movies(movie_name).index.tolist()
    
    content = content_recommendations(movie_name).tolist()
    
    return {
        'Collaborative Filtering': collaborative,
        'Content-Based Filtering': content
    }

In [137]:
hybrid_recommendation('Toy Story (1995)')

{'Collaborative Filtering': ['Toy Story 2 (1999)',
  'Jurassic Park (1993)',
  'Independence Day (a.k.a. ID4) (1996)',
  'Star Wars: Episode IV - A New Hope (1977)',
  'Forrest Gump (1994)',
  'Lion King, The (1994)',
  'Star Wars: Episode VI - Return of the Jedi (1983)',
  'Mission: Impossible (1996)',
  'Groundhog Day (1993)',
  'Back to the Future (1985)'],
 'Content-Based Filtering': ['Antz (1998)',
  'Toy Story 2 (1999)',
  'Adventures of Rocky and Bullwinkle, The (2000)',
  "Emperor's New Groove, The (2000)",
  'Monsters, Inc. (2001)',
  'Wild, The (2006)',
  'Shrek the Third (2007)',
  'Tale of Despereaux, The (2008)',
  'Asterix and the Vikings (Astérix et les Vikings) (2006)',
  'Turbo (2013)']}

In [138]:
import requests

API_KEY = "39f25a5a3323ef3ddea971b643b6459a"

def fetch_poster(movie_name):

    url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={movie_name}"

    response = requests.get(url, timeout=10)

    data = response.json()

    if 'results' not in data or len(data['results']) == 0:
        return "No poster found"

    poster_path = data['results'][0].get('poster_path')

    if poster_path is None:
        return "Poster not available"

    full_path = "https://image.tmdb.org/t/p/w500/" + poster_path

    return full_path

In [139]:
fetch_poster('Toy Story')

'https://image.tmdb.org/t/p/w500//uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg'

In [140]:
import pickle
pickle.dump(similarity_df, open('similarity.pkl', 'wb'))

In [141]:
pickle.dump(movies, open('movies.pkl', 'wb'))